# Vesuvius Surface Detection — Dataset Exploration

**Competition:** [vesuvius-challenge-surface-detection](https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection)

This notebook explores **3D CT volumes** and label masks (`0=bg`, `1=surface`, `2=ignore`).
It does **not** train models.

Schema: [`docs/dataset_schema.md`](../docs/dataset_schema.md)

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "datasets").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.dataset_analysis import SurfaceDatasetInspector
from analysis.modeling_insights import (
    compute_imbalance_report_3d,
    compute_scroll_domain_shift,
    compute_surface_thickness,
    validation_strategy_recommendation,
)
from analysis.utils import ensure_figures_dir, set_publication_style
from analysis import visualization as viz
from datasets import PatchConfig3D, SurfacePatchDataset, build_transforms, validate_dataset

set_publication_style()
FIGURES = ensure_figures_dir()
DATA_ROOT = Path(r"C:\Users\vigne\Downloads\vesuvius-challenge-surface-detection")
if not (DATA_ROOT / "train.csv").exists():
    DATA_ROOT = REPO_ROOT / "data"

inspector = SurfaceDatasetInspector(data_root=DATA_ROOT, split="train")
VOL_ID = inspector.default_volume_id()
print("DATA_ROOT:", DATA_ROOT)
print("volumes:", len(inspector.volume_ids))
print("scrolls:", inspector.scroll_ids)
print("default:", VOL_ID)

---
## Part 1 — Schema & Validation

### What / why
Confirm CSV ↔ image ↔ label identity, shape match, and label values in `{0,1,2}` before any modeling.

### Modeling impact
Hard failures here break nnU-Net conversion and patch training silently later.

In [ ]:
report = inspector.validate()
print(report.summary())
display(report.inventory.head())
assert report.ok, "Fix dataset errors before continuing"

---
## Part 2 — Inventory & Metadata

### What / why
Volume shapes, dtypes, disk size, class fractions, and `scroll_id` grouping.

### Modeling impact
Variable shapes force 3D patching / sliding-window inference. Scroll groups define honest validation.

In [ ]:
print(inspector.overview_text())
inv = inspector.inventory()
display(inv)
scrolls = inspector.scroll_summary()
display(scrolls)
fig = viz.plot_volume_inventory(inv, save_name="01_volume_inventory")
plt.show()
fig_s = viz.plot_scroll_summary(scrolls, save_name="05_scroll_summary")
plt.show()

---
## Part 3 — Orthogonal Volume Views

### What / why
Axial / coronal / sagittal mid-planes with surface (red) and ignore (blue) overlays.

### Modeling impact
Surfaces are thin 3D sheets — 2D-only models miss continuity along the third axis.

In [ ]:
image = inspector.load_image(VOL_ID)
label = inspector.load_label(VOL_ID)
fig = viz.plot_orthogonal_views(image, label, volume_id=VOL_ID, save_name=f"02_orthogonal_{VOL_ID}")
plt.show()
fig_c = viz.plot_class_histogram(label, volume_id=VOL_ID, save_name=f"03_class_hist_{VOL_ID}")
plt.show()

---
## Part 4 — Intensity Statistics (Labeled Voxels)

### What / why
CT intensity distribution **excluding ignore (2)**.

### Modeling impact
Normalization and clipping should be estimated on labeled voxels, not ignore-filled empty space.

In [ ]:
stats = inspector.intensity_stats(VOL_ID)
display(stats.to_series())
fig = viz.plot_intensity_hist(image, label, volume_id=VOL_ID, save_name=f"04_intensity_{VOL_ID}")
plt.show()

---
## Part 5 — Surface Thickness & Patch Scale

### What / why
3D distance-transform thickness of class-1 sheets.

### Modeling impact
Patch depth and encoder resolution must cover several thickness scales.

In [ ]:
thick = compute_surface_thickness(label)
display(pd.Series(thick.to_dict(), name=VOL_ID))
PATCH = (thick.recommended_patch_d,) * 3
STRIDE = tuple(max(p // 2, 8) for p in PATCH)
print("suggested patch_size", PATCH, "stride", STRIDE)

---
## Part 6 — 3D Patch Sampling

### What / why
Valid 3D patches after ignore-aware filtering; montage of axial mid-planes.

### Modeling impact
This is the actual training distribution for a patch-based or nnU-Net-style trainer.

In [ ]:
patch_ds = SurfacePatchDataset(
    root=DATA_ROOT,
    split="train",
    patch_config=PatchConfig3D(
        patch_size=PATCH,
        stride=STRIDE,
        min_labeled_ratio=0.1,
        min_foreground_ratio=0.0,
    ),
    transform=build_transforms("train"),
    normalize="zscore",
    volume_ids=[VOL_ID],
)
print("n_patches", len(patch_ds))
samples = [patch_ds[i] for i in range(min(4, len(patch_ds)))]
fig = viz.plot_patch_montage(samples, save_name=f"06_patches_{VOL_ID}")
plt.show()

---
## Part 7 — Imbalance & Loss Design Signals

### What / why
Voxel neg:pos among labeled voxels; patch surface-fraction histogram; suggested BCE `pos_weight`.

### Modeling impact
Ignore (2) must not enter the ratio. Sparse surface patches need Dice / sampling, not naïve BCE.

In [ ]:
imb, fracs = compute_imbalance_report_3d(label, patch_size=PATCH, stride=STRIDE)
display(pd.Series(imb.to_dict(), name="imbalance"))
fig = viz.plot_imbalance_hist(fracs, imb, volume_id=VOL_ID, save_name=f"07_imbalance_{VOL_ID}")
plt.show()

---
## Part 8 — Scroll Domain Shift & Validation

### What / why
Pairwise scroll intensity histogram distances and surface-density gaps.

### Modeling impact
Prefer **leave-one-scroll-out**. Random patch splits leak via overlap and same-scroll correlation.

In [ ]:
scroll_payload = {}
for sid in inspector.scroll_ids:
    vids = [r.volume_id for r in inspector.records if r.scroll_id == sid]
    vid = vids[0]
    scroll_payload[sid] = {
        "image": inspector.load_image(vid),
        "label": inspector.load_label(vid),
    }

shift_df = compute_scroll_domain_shift(scroll_payload)
display(shift_df)
fig = viz.plot_domain_shift(shift_df, save_name="08_scroll_domain_shift")
plt.show()
print(validation_strategy_recommendation(shift_df))

---
## Part 9 — Research Synthesis

| Decision | Evidence |
|---|---|
| 3D patches not 2.5D slices | Thin sheets + topology metrics |
| Ignore label 2 in loss/metrics | Schema + class histograms |
| Patch size | Thickness analysis |
| Sampler / loss | Patch surface fractions |
| Validation unit | Scroll domain shift |
| nnU-Net later | Same case IDs; ignore=2 |

Next: design model / nnU-Net plans only after this notebook runs cleanly on full data.

---
## Part 10 — Export Audit

In [ ]:
exported = sorted(p for p in FIGURES.glob("*") if p.name != ".gitkeep")
print(f"Exported {len(exported)} artifacts to {FIGURES}")
for p in exported:
    print(f"  {p.name}")